> ## &#9989; SOLUTION / ANSWER KEY &mdash; Lab 1.12
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-12-capstone-model-to-agent.ipynb`](../lab-12-capstone-model-to-agent.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 1.12 &mdash; Capstone: From Model to Agent (LLM + Tools + Goal + Loop)

**Level:** 🔴 Advanced &nbsp;|&nbsp; **Est. time:** 45 min &nbsp;|&nbsp; **Day 1 &middot; Module 1 &mdash; Understanding AI &amp; its Evolution**

### What you'll do
- Assemble the agent pattern: a planner + tools + a goal + a loop
- Implement tool dispatch and the act-observe loop
- Run a multi-step goal end-to-end -- no LLM key required
- (Optional) swap in a real local LLM via Ollama

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

**Reference:** [Module 1 slides &mdash; From a model that answers to an agent that acts](../../presentation/day1-module1-understanding-ai-and-its-evolution.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 1 labs](./index.html)

In [ ]:
# Setup -- run me first
import os
WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-01-12")
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

## Concept &mdash; the destination of the whole course
An **agent = brain (planner) + tools + a goal + a loop**. Today the "brain" is a tiny rule-based
planner so the lab runs anywhere; from Day 3 you swap in a real LLM (LangChain). The *shape* is
identical: read the goal &rarr; plan steps &rarr; **use a tool** &rarr; observe &rarr; loop &rarr; answer.

In [ ]:
# DEMO -- the tools available to our agent
import re

def tool_calc(expr):
    if not re.fullmatch(r"[\d+\-*/(). ]+", expr):
        raise ValueError("unsafe expression")
    return eval(expr, {"__builtins__": {}}, {})

def tool_wordcount(text):
    return len(text.split())

print("calc 3*14+8 =", tool_calc("3*14+8"))
print("wordcount   =", tool_wordcount("the quick brown fox"))

## Your Turn
Wire up the **tool dispatcher** and the **agent loop**. A `plan` is a list of `(tool, argument)`
steps; the agent runs each step, *observes* the result, and returns the **final** observation.

In [ ]:
TOOLS = {'calc': tool_calc, 'wordcount': tool_wordcount}

def run_tool(name, arg):
    """Look up a tool by name and call it with arg."""
    return TOOLS[name](arg)

def agent_run(plan, verbose=True):
    """Execute each (tool, arg) step; observe; return the LAST observation."""
    observation = None
    for step_no, (tool, arg) in enumerate(plan, 1):
        observation = run_tool(tool, arg)
        if verbose:
            print(f'  step {step_no}: {tool}({arg!r}) -> {observation}')
    return observation

# a 2-step goal: 3*14, then add 8 to it
demo_plan = [('calc', '3*14'), ('calc', '42+8')]
try:
    print('FINAL:', agent_run(demo_plan))
except Exception as e: print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("run_tool calls calc", lambda: run_tool("calc", "2+2") == 4)
expect_true("run_tool calls wordcount", lambda: run_tool("wordcount", "a b c") == 3)
expect_true("agent runs a multi-step plan", lambda: agent_run([("calc","3*14"),("calc","42+8")], verbose=False) == 50)
expect_true("agent handles a wordcount goal", lambda: agent_run([("wordcount","the quick brown fox")], verbose=False) == 4)

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

## Optional &mdash; use a REAL local LLM as the planner (Ollama)
If you have [Ollama](https://ollama.com) running with a small model, this turns the goal from a
hand-written plan into one an LLM proposes. Safe to skip &mdash; **not graded**.

In [ ]:
# OPTIONAL: requires `pip install langchain-community` and a running Ollama (llama3.2:1b)
try:
    from langchain_community.chat_models import ChatOllama
    llm = ChatOllama(model="llama3.2:1b", temperature=0)
    msg = llm.invoke("In one short sentence, what is an AI agent?")
    print("LLM says:", getattr(msg, "content", msg))
except Exception as e:
    print("Ollama not available -- that's fine, the lab above is complete without it.")
    print("  reason:", type(e).__name__)

---
### Key takeaway
You built the **LLM + Tools + Goal + Loop** pattern by hand. Every framework this week (LangChain, AutoGPT) is this same loop, scaled up. You're ready for Day 2.

[&#8592; All Module 1 labs](./index.html) &nbsp;&middot;&nbsp; [Module 1 slides](../../presentation/day1-module1-understanding-ai-and-its-evolution.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>